# ESS BEER model

Example of using the `ess_beer_model` package to generate BEER configuration tables and simulation files.

**Repository**: `https://github.com/saroun/ess_beer_model.git`

**Installation**: from the repository root, type `pip install -e .`


In [ ]:
import beer
import beer.geometry as bg
import numpy as np
deg = np.pi/180

## Inspect beam geometry - example

Get beam axis in requested coordiante system:

`TCS` target coordinations  
`FP` W2 beamport coordinates  
`ISCS` Instrument coordinates (as used in siumlations)  


In [ ]:
# Define distance from ISCS origin
L_ISCS = 150000

print('Evaluation of beam axis coordinates at L={:g} mm:'.format(L_ISCS))
# Get beam axis coordinates for given distance. The result is converted 
# to required coordinate system (ISCS, TCS or FP). Distance is always defined as x_ISCS.
r_ISCS = bg.beamAxis(L_ISCS, coord='ISCS')
r_FP = bg.beamAxis(L_ISCS, coord='FP')
r_TCS = bg.beamAxis(L_ISCS, coord='TCS')
print('r_ISCS = {}'.format(r_ISCS)) 
print('r_W2 = {}'.format(r_FP))
print('r_TCS = {}'.format(r_TCS))

# get beam axis angle in [rad] with respect to x-axis of given coordinate system
# NOTE: Fp stands for the beam port W2 coordinates
angle_W2 = bg.beamAngle(L_ISCS, coord='FP')
angle_ISCS = bg.beamAngle(L_ISCS, coord='ISCS')
print('angle_W2 = {:g} deg'.format(angle_W2/deg)) 
print('angle_ISCS = {:g} deg'.format(angle_ISCS/deg)) 

# get beam size at given distance: left, right, top, bottom, width and height, converted to required coordinates.
[left, right, top, bottom, width, height] = bg.beamSize(L_ISCS, coord='FP')[0]
print('Beam size: {:g} x {:g}'.format(width,height))
# convert a point from ISCS to TCS coordinates
r_TCS = bg.ISCS2TCS(r_ISCS)
# or to FP (W2 focal point) coordinates
r_FP = bg.ISCS2FOC(r_ISCS)


## Export configuration tables for BEER 
  
`BEER_components.txt`, the contents of the BEER optics table, ESS-0478295.  
`BEER_beam.txt`, a table with beam axis coordinates in three reference frames (TCS, W2, ISCS).  
`BEER_modes.txt`, reference operation modes with corresponding components settings. 

The tables are text files with tab separated columns. The BEER optics specification table (ESS-0478295) can be updated by copy and paste of `BEER_components.txt` into the excel sheet.

In [ ]:
beer.reportTables(outdir='') # specify output directory

## Choppers
Calculate and plot chopper transmission functions for various operation modes:

In [ ]:
beer.reportChoppers(outdir='')

## Guide geometry
Plot profiles of neutron guides in given coordinate system. Options are: ISCS or FP (W2):

In [ ]:
beer.plotGuides(outdir='', coord='FP')

## McStas instrument file
Generate McStas instrument file for primary beam line (source to sample slit). The generated `BEER_reference.instr` file can be used to run simulations of the reference operation modes as specified in the *BEER optics report* (ESS-0238217).


In [ ]:
import beer.mcstas as mcstas

# Create instrument file for given McStas version (2 or 3) in given target folder. 
# NOTE: without specifying these arguments, McStas and worspace paths should be configured first.  

mcstas.create_instrument(version=3, outpath='.')


## SIMRES configuration

The reference configuration file for SIMRES (see https://github.com/saroun/simres) is available in the package resources (`beer/resources/BEER_reference.xml`). The following command generates the input script for SIMRES, which sets the instrument parameters according to the current BEER geometry definition (equivalent to the McStas model).


In [ ]:
import beer.simres as simres
simres.create_setup_script(outpath='.')


### Script for running reference modes

For the mode names, see the table of modes generated by beer.reportTables().

In [ ]:
simres.create_run_script('PS2', outpath='.', waviness=0.1)